# 02 · İlk yarış sıralaması tahmini

**Hedef:** bir hipotezi çalışır tahmine dönüştürmek, kaydetmek ve hatasını yorumlamak.

İlk hipotez: **Q'daki sıra yarış için makul bir başlangıçtır.** Bu eğitilmiş ML modeli değil; ilerideki modellerin geçmesi gereken benchmark.
2024 Bahreyn'i geçmişten yeniden kuruyoruz; canlı/as-of performans kanıtı değil.

## Kurulum

01 defterindeki veri ve alıştırmayı tamamladıktan sonra ilerle.
Kernel: **WeAreChecking (.venv)**. Yarış sonucu girdi tablosuna eklenmeyecek.

In [ ]:
import importlib
from IPython.display import display
from wearechecking.paths import project_root
from wearechecking.data import latest_snapshot
from wearechecking.features import qualifying_inputs
from wearechecking.evaluation import save_prediction, evaluate_saved
from wearechecking.learning import check_baseline
from wearechecking import student

ROOT = project_root()
snapshot = latest_snapshot(2024, 1, root=ROOT)
CUTOFF = '2024-03-02T12:00:00Z'

## Adımlar

### 1. Tahmin anını seç

Kesim zamanı Q bitişinden sonra, yarış başlangıcından önce.
Veri bugün indirildiğinden sonradan yapılmış Q düzeltmeleri bulunabilir; `reconstructed` bunu belirtir.
Q katılımcıları burada geçici yarış kadrosudur. Ceza/substitution/DNS sonraki aşamanın konusu.

In [ ]:
inputs = qualifying_inputs(snapshot, CUTOFF, mode='reconstructed')
features = inputs.frame
print('Kesim zamanı:', inputs.metadata['cutoff_utc'])
print('Girdi kolonları:', features.columns.tolist())
display(features.head(8))

### 2. Hipotezi kodla: qualifying_baseline

`src/wearechecking/student.py` içindeki ikinci fonksiyonu düzenle.
Girdi: driver_id, driver, team, qualifying_position.
Çıktı: **driver_id, predicted_position**.

İpuçları:
1. Küçük qualifying_position daha iyi sıradır.
2. Gerekli iki kolonu seçip kopya oluştur.
3. Sırala; tahmin kolonuna beklenen adı ver.
4. Sürücü sayısını 20 diye sabitleme.

**Önce düşün:** R seansındaki Position kolonunu kullansaydık neden hile yapmış olurduk?

In [ ]:
importlib.reload(student)
prediction = student.qualifying_baseline(features.copy())
print(check_baseline(prediction, features))
if prediction is not None:
    display(prediction.sort_values('predicted_position').head(10))

### 3. Tahmini sabitle

Yarış sonucuyla kıyaslamadan önce bir tahmin kaydı oluşturuyoruz.
Her çalıştırmada yeni kayıt; eski tahminin üzerine yazılmaz.

In [ ]:
prediction_folder = None
if prediction is None:
    print('BEKLİYOR: önce tahmin fonksiyonunu tamamla.')
else:
    check_baseline(prediction, features)
    prediction_folder = save_prediction(prediction, inputs, root=ROOT)
    print('Kaydedildi:', prediction_folder.name)

### 4. Şimdi gerçek sonucu aç

**MAE:** tahmin sırası ile gerçek sıra arasındaki mutlak farkların ortalaması; düşük daha iyi.
**Spearman:** sıralamaların ilişkisi; +1 aynı sıra, -1 ters sıra.

Tek geçmiş yarış genel başarı kanıtı değildir. İleride çok yarışlı zaman sıralı test gerekir.
Sınıflandırılmış DNF'nin resmi sırası korunur; durum kolonu gösterilir. Farklı/eksik kadro sessizce düzeltilmez.

In [ ]:
if prediction_folder is None:
    print('BEKLİYOR: kaydedilmiş tahmin yok; sonuç tablosu henüz açılmadı.')
else:
    comparison, metrics = evaluate_saved(prediction_folder, snapshot)
    display(comparison.sort_values('absolute_error', ascending=False).head(10))
    print('Sıra MAE:', round(metrics['position_mae'], 3))
    print('Spearman:', round(metrics['spearman'], 3))

## Kontroller

- Her girdi sürücüsü için bir tahmin var; sıra 1..N.
- Yarış sonuçları yalnız tahmin kaydedildikten sonra açıldı.
- Kontrol **TAMAM** diyor; skorları kendi çıktımdan okuyabiliyorum.
- Tek yarış skorunun neden genellenemeyeceğini açıklayabiliyorum.

## Sonraki adım

`docs/DECISION_NOTE.md` dosyasını kendi çıktınla doldur.
İlk geliştirme için bir fikir seç: takım geçmişi, sürücü geçmişi veya FP temposu. Önce adil değerlendirmeyi konuşacağız.